In [3]:
# import neccesary labarary
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, Nadam
# Pre Trained Models
import tensorflow_hub as hub

## Load DataSet

In [6]:
# load tensorflow dataset
import tensorflow_datasets as tfds

raw_train_set, raw_valid_set, raw_test_set = tfds.load(
    name="imdb_reviews",
    split=["train[:90%]", "train[90%:]", "test"],
    as_supervised=True)

In [7]:
# show x, y in data 
for X, y in raw_train_set.take(1): # generator
    print(f"X: {X}, y: {y}")

X: b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.", y: 0


In [8]:
# split data to batch
tf.random.set_seed(42)

train_ds = raw_train_set.shuffle(5000, seed=42).batch(32).prefetch(1)
valid_ds = raw_valid_set.batch(32).prefetch(1)
test_ds = raw_test_set.batch(32).prefetch(1)

In [9]:
for X_batch, y_batch in train_ds.take(1):
    print(f"X shape: {X_batch.shape}")
    print(f"y shape: {y_batch.shape}")

X shape: (32,)
y shape: (32,)


In [10]:
X.numpy().decode().split(" ")

['This',
 'was',
 'an',
 'absolutely',
 'terrible',
 'movie.',
 "Don't",
 'be',
 'lured',
 'in',
 'by',
 'Christopher',
 'Walken',
 'or',
 'Michael',
 'Ironside.',
 'Both',
 'are',
 'great',
 'actors,',
 'but',
 'this',
 'must',
 'simply',
 'be',
 'their',
 'worst',
 'role',
 'in',
 'history.',
 'Even',
 'their',
 'great',
 'acting',
 'could',
 'not',
 'redeem',
 'this',
 "movie's",
 'ridiculous',
 'storyline.',
 'This',
 'movie',
 'is',
 'an',
 'early',
 'nineties',
 'US',
 'propaganda',
 'piece.',
 'The',
 'most',
 'pathetic',
 'scenes',
 'were',
 'those',
 'when',
 'the',
 'Columbian',
 'rebels',
 'were',
 'making',
 'their',
 'cases',
 'for',
 'revolutions.',
 'Maria',
 'Conchita',
 'Alonso',
 'appeared',
 'phony,',
 'and',
 'her',
 'pseudo-love',
 'affair',
 'with',
 'Walken',
 'was',
 'nothing',
 'but',
 'a',
 'pathetic',
 'emotional',
 'plug',
 'in',
 'a',
 'movie',
 'that',
 'was',
 'devoid',
 'of',
 'any',
 'real',
 'meaning.',
 'I',
 'am',
 'disappointed',
 'that',
 'there',


In [9]:
len(X.numpy().decode().split(" ")) # one raw contain 116 word

116

In [10]:
# count all word in train and vaild and test data
counter = 0
for X, y in raw_test_set:
    counter += 1
counter # Train Set 22_500, Valid Set 2_500, Test Set 25_000

25000

## NLP MODELING

In [11]:
# Word Level
max_tokens = 1000 # 0 -> 999 {0: pad, 1: unk, 2:}

tokenizer = TextVectorization(
    max_tokens = max_tokens,
    standardize = 'lower_and_strip_punctuation',
    split = 'whitespace')

In [12]:
tokenizer.adapt([X.numpy().decode() for X, y in raw_train_set])

In [13]:
# Encoding
token_ids = tokenizer('I love you') 
token_ids

<tf.Tensor: shape=(3,), dtype=int64, numpy=array([ 10, 115,  23])>

In [14]:
# Decoding
def decode_to_text(token_ids, tokenizer):
    text = ""
    for index in token_ids:
        text = text +f" {tokenizer.get_vocabulary()[index]}"
    text = text.strip()
    return text

In [15]:
decode_to_text(token_ids, tokenizer)

'i love you'

## Model Arch

In [16]:
# Arch1
max_tokens = 1000  # 0 -> 999 {0: pad, 1: unk, 2:}

input_ = Input(shape=(), dtype=tf.string)
hidden = tokenizer(input_)
hidden = Embedding(input_dim=max_tokens, output_dim=128, mask_zero=True)(hidden)
hidden = SimpleRNN(32, activation='tanh', return_sequences = True, kernel_initializer='glorot_uniform')(hidden)
hidden = SimpleRNN(32, activation='tanh', kernel_initializer='glorot_uniform')(hidden)
output = Dense(1, activation='sigmoid', kernel_initializer='glorot_uniform')(hidden)

imdb_sentiment_analysis = Model(inputs=[input_], outputs=[output])
imdb_sentiment_analysis.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, None)      │          0 │ input_layer[0][0] │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │    128,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, None, 32)  │      5,152 │ embedding[0][0],  │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_1        │ (None, 32)        │      2,080 │ simple_rnn[0][0], │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │         33 │ simple_rnn_1[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 135,265 (528.38 KB)

 Trainable params: 135,265 (528.38 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile
optimizer = Adam(learning_rate=1e-3)
imdb_sentiment_analysis.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
# Training
history = imdb_sentiment_analysis.fit(train_ds, validation_data=valid_ds, epochs=5)

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1685s 2s/step - accuracy: 0.6991 - loss: 0.5771 - val_accuracy: 0.7444 - val_loss: 0.5280
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1702s 2s/step - accuracy: 0.5945 - loss: 0.6619 - val_accuracy: 0.5476 - val_loss: 0.6803
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1673s 2s/step - accuracy: 0.5940 - loss: 0.6612 - val_accuracy: 0.5768 - val_loss: 0.6669
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6204 - loss: 0.6406

### Train Set => 22_500 (so low to train good embedding matrix) 
* so that load pretrained Embedding Layer

## pretrained Embedding Layer

In [17]:
trainable = False
pretrained_use_embedding_layer = hub.KerasLayer("https://www.kaggle.com/models/google/universal-sentence-encoder/TensorFlow2/universal-sentence-encoder/2", trainable=trainable)

embedding_vector = pretrained_use_embedding_layer([
    "The quick brown fox jumps over the lazy dog.",
    "I am a sentence for which I would like to get its embedding"])

embedding_vector

<tf.Tensor: shape=(2, 512), dtype=float32, numpy=
array([[-0.03133018, -0.06338634, -0.016075  , ..., -0.0324278 ,
        -0.04575739,  0.05370454],
       [ 0.0508086 , -0.01652432,  0.01573776, ...,  0.00976658,
         0.0317012 ,  0.01788118]], dtype=float32)>

In [18]:
# Transfer to keras layers
class USE_Embedding(tf.keras.layers.Layer):

    def __init__(self, link, trainable):
        super().__init__()
        self.embedding_layer = hub.KerasLayer(link, trainable=trainable)
        
    def call(self, inputs):
        return self.embedding_layer(inputs)

In [23]:
# Arch 2
# Transfer Learning
trainable = False
pretrained_use_embedding_layer = USE_Embedding("https://www.kaggle.com/models/google/universal-sentence-encoder/TensorFlow2/universal-sentence-encoder/2", trainable=trainable)

input_ = Input(shape=(), dtype=tf.string)
hidden = pretrained_use_embedding_layer(input_) # 'This is a sentence' = [512]
hidden = Dense(32, activation='relu', kernel_initializer='he_uniform')(hidden)
output = Dense(1, activation='sigmoid', kernel_initializer='glorot_uniform')(hidden)

imdb_sentiment_analysis_use_pretrained = Model(inputs=[input_], outputs=[output])
imdb_sentiment_analysis_use_pretrained.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None)                 │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ use__embedding (USE_Embedding)  │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │        16,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,449 (64.25 KB)

 Trainable params: 16,449 (64.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# compile & Traning
optimizer = Nadam(learning_rate=1e-3)
imdb_sentiment_analysis_use_pretrained.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
history = imdb_sentiment_analysis_use_pretrained.fit(train_ds, validation_data=valid_ds, epochs=5)

## Load Word-Level Pretrained Embedding (Glove)

In [19]:
# create Embedding Matrix
import numpy as np
embedding_dim = 100

vocab = tokenizer.get_vocabulary()
word_index = dict(zip(vocab, range(len(vocab))))

embedding_matrix = np.zeros(
    (len(vocab), embedding_dim),
    dtype="float32"
)

found_words = set()

with open(
    "/kaggle/input/models/rudra29/glove/keras/default/1/glove.6B.100d.txt", encoding="utf8"
         ) as f:

    for line in f:
        values = line.split()
        word = values[0]

        if word in word_index:
            vector = np.asarray(values[1:], dtype="float32")
            embedding_matrix[word_index[word]] = vector
            found_words.add(word)

no_glove_words = [
    word for word in vocab
    if word not in found_words
]

print("Embedding matrix shape:", embedding_matrix.shape)
print("Words without GloVe:", len(no_glove_words))

Embedding matrix shape: (1000, 100)
Words without GloVe: 7


### Note
1. Tokenizer
       ↓
word → index

2. GloVe
       ↓
word → vector

3. Embedding Matrix
       ↓
index → vector

In [30]:
# Arch 3
input_ = Input(shape=(), dtype=tf.string)
hidden = tokenizer(input_)
hidden = Embedding(input_dim=max_tokens, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False, mask_zero=True)(hidden)
hidden = SimpleRNN(32, activation='tanh', kernel_initializer='glorot_uniform')(hidden)
output = Dense(1, activation='sigmoid', kernel_initializer='glorot_uniform')(hidden)

imdb_sentiment_analysis_glove_pretrained = Model(inputs=[input_], outputs=[output])
imdb_sentiment_analysis_glove_pretrained.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, None)      │          0 │ input_layer_2[0]… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 100) │    100,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_2        │ (None, 32)        │      4,256 │ embedding_1[0][0… │
│ (SimpleRNN)         │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ simple_rnn_2[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 104,289 (407.38 KB)

 Trainable params: 4,289 (16.75 KB)

 Non-trainable params: 100,000 (390.62 KB)

In [31]:
# Compile and Traning 
optimizer = Adam(learning_rate=1e-3)
imdb_sentiment_analysis_glove_pretrained.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
history = imdb_sentiment_analysis_glove_pretrained.fit(train_ds, validation_data=valid_ds, epochs=5)

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 786s 1s/step - accuracy: 0.5477 - loss: 0.6872 - val_accuracy: 0.5868 - val_loss: 0.6712
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 778s 1s/step - accuracy: 0.5540 - loss: 0.6774 - val_accuracy: 0.5228 - val_loss: 0.6894
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 773s 1s/step - accuracy: 0.5460 - loss: 0.6821 - val_accuracy: 0.5488 - val_loss: 0.6801
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 768s 1s/step - accuracy: 0.5611 - loss: 0.6761 - val_accuracy: 0.5668 - val_loss: 0.6742


In [50]:
# Arch 4 LSTM
input_ = Input(shape=(), dtype=tf.string)
hidden = tokenizer(input_)
hidden = Embedding(input_dim=max_tokens, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False, mask_zero=True)(hidden)
hidden = LSTM(100, activation='tanh', kernel_initializer='glorot_uniform')(hidden)
output = Dense(1, activation='sigmoid', kernel_initializer='glorot_uniform')(hidden)

imdb_sentiment_analysis_glove_pretrained_lstm = Model(inputs=[input_], outputs=[output])
imdb_sentiment_analysis_glove_pretrained_lstm.summary()

Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, None)      │          0 │ input_layer_12[0… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_11        │ (None, None, 100) │    100,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_11        │ (None, None)      │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_9 (LSTM)       │ (None, 100)       │     80,400 │ embedding_11[0][… │
│                     │                   │            │ not_equal_11[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 1)         │        101 │ lstm_9[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 180,501 (705.08 KB)

 Trainable params: 80,501 (314.46 KB)

 Non-trainable params: 100,000 (390.62 KB)

In [46]:
# Compile and Traning
optimizer = Adam(learning_rate=1e-3)
imdb_sentiment_analysis_glove_pretrained_lstm.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
history = imdb_sentiment_analysis_glove_pretrained_lstm.fit(train_ds, validation_data=valid_ds, epochs=5)

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 22s 28ms/step - accuracy: 0.6565 - loss: 0.6121 - val_accuracy: 0.7916 - val_loss: 0.4569
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - accuracy: 0.8004 - loss: 0.4350 - val_accuracy: 0.8204 - val_loss: 0.3920
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - accuracy: 0.8240 - loss: 0.3911 - val_accuracy: 0.8432 - val_loss: 0.3627
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - accuracy: 0.8355 - loss: 0.3685 - val_accuracy: 0.8428 - val_loss: 0.3487
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - accuracy: 0.8437 - loss: 0.3522 - val_accuracy: 0.8556 - val_loss: 0.3317


In [20]:
# Arch 5 Bidirectional LSTM
input_ = Input(shape=(), dtype=tf.string)
hidden = tokenizer(input_)
hidden = Embedding(input_dim=max_tokens, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False, mask_zero=True)(hidden)
hidden = Bidirectional(LSTM(100, activation='tanh', kernel_initializer='glorot_uniform'))(hidden)
output = Dense(1, activation='sigmoid', kernel_initializer='glorot_uniform')(hidden)

imdb_sentiment_analysis_glove_pretrained_Bi = Model(inputs=[input_], outputs=[output])
imdb_sentiment_analysis_glove_pretrained_Bi.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, None)      │          0 │ input_layer_1[0]… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 100) │    100,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 200)       │    160,800 │ embedding_1[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        201 │ bidirectional[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 261,001 (1019.54 KB)

 Trainable params: 161,001 (628.91 KB)

 Non-trainable params: 100,000 (390.62 KB)

In [21]:
# Compile and Traning
optimizer = Adam(learning_rate=1e-3)
imdb_sentiment_analysis_glove_pretrained_Bi.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
history = imdb_sentiment_analysis_glove_pretrained_Bi.fit(train_ds, validation_data=valid_ds, epochs=5)

Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 39s 51ms/step - accuracy: 0.6600 - loss: 0.6136 - val_accuracy: 0.7696 - val_loss: 0.4901
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 35s 50ms/step - accuracy: 0.7922 - loss: 0.4522 - val_accuracy: 0.7976 - val_loss: 0.4259
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 35s 49ms/step - accuracy: 0.8221 - loss: 0.3944 - val_accuracy: 0.8372 - val_loss: 0.3606
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 35s 49ms/step - accuracy: 0.8408 - loss: 0.3575 - val_accuracy: 0.8404 - val_loss: 0.3566
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 35s 49ms/step - accuracy: 0.8552 - loss: 0.3313 - val_accuracy: 0.8588 - val_loss: 0.3255


## Evalute The Best Model

In [23]:
# Evalute
imdb_sentiment_analysis_glove_pretrained_Bi.evaluate(test_ds)

782/782 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.8534 - loss: 0.3332


[0.3332076668739319, 0.8533599972724915]

## Save Model

In [48]:
# save model
imdb_sentiment_analysis_glove_pretrained_Bi.save("Bidirectional_Lstm_model.h5")

In [29]:
# save as a keras model
model_path = "/kaggle/working/Bidirectional_Lstm_model.keras"

imdb_sentiment_analysis_glove_pretrained_Bi.save(model_path)

print(f"Model saved to: {model_path}")

Model saved to: /kaggle/working/Bidirectional_Lstm_model.keras


In [30]:
import os

os.listdir("/kaggle/working")

['Bidirectional_Lstm_model.keras',
 '.virtual_documents',
 'Bidirectional_Lstm_model.h5']

In [35]:
from tensorflow.keras.models import load_model

loaded_model = load_model("/kaggle/working/Bidirectional_Lstm_model.keras")
loaded_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, None)      │          0 │ input_layer_1[0]… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 100) │    100,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 200)       │    160,800 │ embedding_1[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        201 │ bidirectional[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 422,004 (1.61 MB)

 Trainable params: 161,001 (628.91 KB)

 Non-trainable params: 100,000 (390.62 KB)

 Optimizer params: 161,003 (628.92 KB)

In [44]:
label_map = {
    0: "Bad Review",
    1: "Good Review" }

In [45]:
# loaded model and make prediction
for x, y in test_ds.take(1):
    predictions = loaded_model.predict(x)
    predicted_classes = (predictions > 0.5).astype(int)
    results = [label_map[int(x)] for x in predicted_classes.flatten()]

    print("Actual:   ", y.numpy())
    print("Predicted:", predicted_classes.flatten())

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Actual:    [1 1 0 0 1 1 1 1 0 1 0 0 1 0 1 0 1 0 1 0 0 1 0 0 1 1 0 0 0 1 1 1]
Predicted: [1 1 0 0 1 1 1 1 0 1 0 1 0 0 1 1 1 1 1 0 0 1 0 0 1 0 0 0 0 1 1 1]
